# Customizing the density estimator


`sbi` allows to specify a specific density estimator for each of the implemented methods.
We support a variety of density estimators, e.g., mixtures of Gaussians, normalizing
flows, and diffusion models. Some of the density estimators are implemented as part of
`sbi`, for others we rely on other packages like
[`nflows`](https://github.com/bayesiains/nflows/) or [`zuko`](https://github.com/probabilists/zuko). 

For all options, check the API reference
[here](https://sbi.readthedocs.io/en/latest/sbi.html#neural-nets).

## Changing the type of density estimator

The density estimator is chosen by passing a **config object** in
the `density_estimator` keyword argument to the inference object (`NPE` or `NLE`). There is one config class per model, so the choice of model is the choice of class, e.g.
`MAFConfig` for a Masked Autoregressive Flow, or `NSFConfig` for a Neural Spline Flow with default
hyperparameters. The config carries only the settings that model accepts, so an unsupported or misspelled one raises at construction instead of being ignored.

Note that `MAFConfig` or `NSFConfig` correspond to `nflows` density
estimators. Those have proven to work well, but the `nflows` package is not maintained
anymore. To use more recent and actively maintained density estimators, we tentatively
recommend using `zuko`, e.g., `ZukoMAFConfig` or `ZukoNSFConfig`. 


In [1]:
import torch

from sbi.inference import NPE, NRE
from sbi.utils import BoxUniform

In [2]:
from sbi.neural_nets import ZukoMAFConfig

prior = BoxUniform(torch.zeros(2), torch.ones(2))
inference = NPE(prior=prior, density_estimator=ZukoMAFConfig())

In the case of `NRE`, the argument is called `classifier`:


In [3]:
from sbi.neural_nets import ResNetClassifierConfig

inference = NRE(prior=prior, classifier=ResNetClassifierConfig())

## Changing hyperparameters of density estimators


The hyperparameters of a model are the constructor arguments of its config, so you can tune them for the problem at hand.

Here, because we want to use N*P*E, we pass the config to the `density_estimator` argument of `NPE`. In this example, we will create a neural spline flow (`ZukoNSFConfig`) with `60` hidden units and `3` transform layers:


In [4]:
# The same configs are used for NLE. NRE takes classifier configs instead.
from sbi.neural_nets import ZukoNSFConfig

density_estimator = ZukoNSFConfig(hidden_features=60, num_transforms=3)
inference = NPE(prior=prior, density_estimator=density_estimator)

It is also possible to pass an `embedding_net` to a config to automatically
learn summary statistics from high-dimensional simulation outputs. You can find a more
detailed tutorial on this in [04_embedding_networks](https://sbi.readthedocs.io/en/latest/how_to_guide/04_embedding_networks.html).


## Building new density estimators from scratch


Finally, it is also possible to implement your own density estimator from scratch, e.g., including embedding nets to preprocess data, or to a density estimator architecture of your choice.

For this, the `density_estimator` argument needs to be a function that takes `theta` and `x` batches as arguments to then construct the density estimator after the first set of simulations was generated. Our factory functions in `sbi/neural_nets/factory.py` return such a function.

The returned `density_estimator` object needs to be a subclass of [`DensityEstimator`](https://github.com/sbi-dev/sbi/blob/1928f018fa08bb0c5309a34d8e95b9f2916b20a5/sbi/neural_nets/estimators/base.py#L11), which requires to implement three methods:
    
- `log_prob(input, condition, **kwargs)`: Return the log probabilities of the inputs given a condition or multiple i.e. batched conditions.
- `loss(input, condition, **kwargs)`: Return the loss for training the density estimator.
- `sample(sample_shape, condition, **kwargs)`: Return samples from the density estimator.